# ARIMA

# Setup

In [ ]:
!pip install statsmodels mlflow --quiet
import pandas as pd, numpy as np, mlflow, os
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN")
os.environ['MLFLOW_TRACKING_USERNAME'] = 'gdzag22'
os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token
mlflow.set_tracking_uri("https://dagshub.com/Nestor-Dzadzamia/walmart-sales-forecasting.mlflow")
mlflow.set_experiment("ARIMA_Training")

# Shared functions (from EDA notebook)

In [ ]:
def load_and_merge(df, features, stores):
    out = df.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
    out = out.merge(stores, on='Store', how='left')
    out['Date'] = pd.to_datetime(out['Date'])
    return out.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

MD_COLS = [f"MarkDown{i}" for i in range(1, 6)]

def preprocess(df):
    out = df.copy()
    out[MD_COLS] = out[MD_COLS].fillna(0)
    out[["CPI", "Unemployment"]] = out.groupby("Store")[["CPI", "Unemployment"]].ffill()
    return out

def wmae(y_true, y_pred, is_holiday):
    w = np.where(is_holiday, 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)

def coldstart_fallback(df_train, df_test):
    train_pairs = set(zip(df_train.Store, df_train.Dept))
    mask = ~pd.Series(list(zip(df_test.Store, df_test.Dept)), index=df_test.index).isin(train_pairs)
    cold = df_test[mask].copy()
    recent = df_train[df_train['Date'] >= df_train['Date'].max() - pd.Timedelta(weeks=52)]
    med = recent.groupby(['Type', 'Dept'])['Weekly_Sales'].median()
    cold['y_fallback'] = [med.get((t, d), 0.0) for t, d in zip(cold['Type'], cold['Dept'])]
    cold['y_fallback'] = cold['y_fallback'].clip(lower=0)
    return cold

# Load and Merge Data


In [ ]:
path = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/"
train = pd.read_csv(path + "train.csv.zip")
test = pd.read_csv(path + "test.csv.zip")
features = pd.read_csv(path + "features.csv.zip")
stores = pd.read_csv(path + "stores.csv")

df = load_and_merge(train, features, stores)
df_test = load_and_merge(test, features, stores)

print("train merged:", df.shape)
print("test merged:", df_test.shape)

# Cleaning

In [ ]:
with mlflow.start_run(run_name="ARIMA_Cleaning"):
    mlflow.log_param("markdown_fill", "0")
    mlflow.log_param("cpi_unemployment_fill", "ffill_per_store")
    mlflow.log_metric("train_rows_before", len(df))
    mlflow.log_metric("markdown_nan_before", int(df[MD_COLS].isna().sum().sum()))

    df_clean = preprocess(df)

    mlflow.log_metric("train_rows_after", len(df_clean))
    mlflow.log_metric("markdown_nan_after", int(df_clean[MD_COLS].isna().sum().sum()))

print(df_clean.shape)
print("Remaining MarkDown NaNs:", df_clean[MD_COLS].isna().sum().sum())

# Validation Split

In [ ]:
VALIDATION_START = df_test['Date'].min() - pd.Timedelta(weeks=52)
VALIDATION_END = VALIDATION_START + pd.Timedelta(weeks=39)

def temporal_split(df):
    tr = df[df["Date"] < VALIDATION_START]
    va = df[(df["Date"] >= VALIDATION_START) & (df["Date"] < VALIDATION_END)]
    return tr, va

tr, va = temporal_split(df_clean)
print("train:", tr["Date"].max(), tr.shape)
print("val:", va["Date"].min(), va.shape)

# Representative Series Selection

In [ ]:
examples = [(1, 1), (4, 92), (20, 1)]

fig, axes = plt.subplots(len(examples), 1, figsize=(12, 8), sharex=True)
for ax, (s, d) in zip(axes, examples):
    series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
    ax.plot(series['Date'], series['Weekly_Sales'])
    ax.set_title(f"Store {s}, Dept {d}")
plt.tight_layout()
plt.show()

# Stationarity Testing (Augmented Dickey-Fuller)

In [ ]:
def run_adf(series, label):
    result = adfuller(series.dropna())
    print(f"{label}")
    print(f"  ADF statistic: {result[0]:.3f}")
    print(f"  p-value: {result[1]:.4f}")
    print(f"  Stationary (p < 0.05)? {'Yes' if result[1] < 0.05 else 'No'}")
    print()

for s, d in examples:
    series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')['Weekly_Sales']
    run_adf(series, f"Store {s}, Dept {d} (raw)")

# Differencing and Seasonal Differencing

In [ ]:
def run_adf_series(series, label):
    result = adfuller(series.dropna())
    print(f"{label}: ADF stat={result[0]:.3f}, p-value={result[1]:.4f}, stationary={'Yes' if result[1] < 0.05 else 'No'}")

s, d = examples[1] 
series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')['Weekly_Sales'].reset_index(drop=True)

diff1 = series.diff().dropna()
seasonal_diff = series.diff(52).dropna()

run_adf_series(series, "Raw")
run_adf_series(diff1, "After 1st differencing (d=1)")
run_adf_series(seasonal_diff, "After seasonal differencing (D=1, lag 52)")

In [ ]:
combined_diff = series.diff().diff(52).dropna()
run_adf_series(combined_diff, "After BOTH differencing (d=1) and seasonal differencing (D=1, lag 52)")

# ACF and PACF Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(combined_diff, lags=40, ax=axes[0])
axes[0].set_title("ACF after differencing (d=1, D=1, lag 52): Store 4, Dept 92")
plot_pacf(combined_diff, lags=40, ax=axes[1])
axes[1].set_title("PACF after differencing (d=1, D=1, lag 52): Store 4, Dept 92")
plt.tight_layout()
plt.show()

# Fitting ARIMA (Non-Seasonal)

In [ ]:
arima_results = {}

for s, d in examples:
    series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
    y = series['Weekly_Sales'].reset_index(drop=True)

    train_y = y[:-39]
    test_y = y[-39:]

    with mlflow.start_run(run_name=f"ARIMA_Store{s}_Dept{d}"):
        mlflow.log_param("order", "(1,1,1)")
        mlflow.log_param("store", s)
        mlflow.log_param("dept", d)

        model = ARIMA(train_y, order=(1,1,1))
        fitted = model.fit()

        forecast = fitted.forecast(steps=39)
        is_holiday = series['IsHoliday'].values[-39:]
        score = wmae(test_y.values, forecast.values, is_holiday)

        mlflow.log_metric("wmae", score)

    arima_results[f"Store{s}_Dept{d}"] = {'actual': test_y.values, 'forecast': forecast.values, 'wmae': score}
    print(f"Store {s}, Dept {d}: ARIMA(1,1,1) WMAE = {score:.2f}")

In [ ]:
fig, axes = plt.subplots(len(examples), 1, figsize=(12, 9))
for ax, (s, d) in zip(axes, examples):
    key = f"Store{s}_Dept{d}"
    r = arima_results[key]
    ax.plot(r['actual'], label='Actual')
    ax.plot(r['forecast'], label='ARIMA(1,1,1) Forecast')
    ax.set_title(f"Store {s}, Dept {d} — WMAE {r['wmae']:.0f}")
    ax.legend()
plt.tight_layout()
plt.show()

# Pipeline Artifact 

In [ ]:
class ARIMAPipeline(mlflow.pyfunc.PythonModel):
    def __init__(self, train_clean, order=(1,1,1)):
        self.train_clean = train_clean
        self.order = order

    def predict(self, context, model_input):
        test_merged = load_and_merge(model_input.copy(), features, stores)
        test_clean = preprocess(test_merged)

        results = []
        train_pairs = set(zip(self.train_clean.Store, self.train_clean.Dept))
        test_pairs = set(zip(test_clean.Store, test_clean.Dept))
        warm_pairs = test_pairs & train_pairs
        cold_pairs_mask = test_clean.apply(lambda r: (r['Store'], r['Dept']) not in train_pairs, axis=1)
        test_warm = test_clean[~cold_pairs_mask].copy()
        test_cold = test_clean[cold_pairs_mask].copy()

        for (s, d) in warm_pairs:
            series = self.train_clean[(self.train_clean['Store']==s) & (self.train_clean['Dept']==d)].sort_values('Date')
            dates_needed = test_warm[(test_warm['Store']==s) & (test_warm['Dept']==d)]['Date'].sort_values()
            if len(dates_needed) == 0 or len(series) < 30:
                continue
            try:
                model = ARIMA(series['Weekly_Sales'].values, order=self.order)
                fitted = model.fit()
                forecast = fitted.forecast(steps=len(dates_needed))
                for dt, val in zip(dates_needed, forecast):
                    results.append({'Store': s, 'Dept': d, 'Date': dt, 'Weekly_Sales': val})
            except Exception:
                continue

        warm_result = pd.DataFrame(results) if results else pd.DataFrame(columns=['Store','Dept','Date','Weekly_Sales'])
        cold_result = coldstart_fallback(self.train_clean, test_cold)
        cold_result = cold_result[['Store','Dept','Date','y_fallback']].rename(columns={'y_fallback':'Weekly_Sales'})

        final = pd.concat([warm_result, cold_result], ignore_index=True)
        final['Weekly_Sales'] = final['Weekly_Sales'].clip(lower=0)
        return final

In [ ]:
pipeline_arima = ARIMAPipeline(train_clean=df_clean, order=(1,1,1))

demo_test = test[test.apply(lambda r: (r['Store'], r['Dept']) in set(examples), axis=1)].copy()
demo_result = pipeline_arima.predict(None, demo_test)
print(demo_result.shape)
print(demo_result.head())

with mlflow.start_run(run_name="ARIMA_Pipeline_Wrapped"):
    mlflow.log_param("order", str((1,1,1)))
    mlflow.log_param("scope", "validated on 3 representative series only, per assignment instruction to limit time on ARIMA/SARIMA")
    mlflow.pyfunc.log_model(python_model=pipeline_arima, name="pipeline")

**Pipeline Scope Note**: The pipeline above is functionally correct and can
accept raw test data for any (Store, Dept) combination, fitting ARIMA
on-the-fly per series. It was validated only on the 3 representative series
used throughout this notebook, not the full ~3,300 series in the real test
set, per the assignment's explicit instruction to minimize time spent on
ARIMA/SARIMA. A full-scale run would require fitting thousands of individual
models, a significant time cost not justified given ARIMA's already-
demonstrated limitation (no seasonal component) and the availability of
better-suited models (SARIMA, tree-based models) already built.

In [ ]:
with mlflow.start_run(run_name="ARIMA_Pipeline_Wrapped"):
    mlflow.log_param("order", str((1,1,1)))
    mlflow.log_param("scope", "validated on 3 representative series only, per assignment instruction to limit time on ARIMA/SARIMA")
    model_info = mlflow.pyfunc.log_model(python_model=pipeline_arima, name="pipeline")

loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
reloaded_result = loaded_model.predict(demo_test)

print("Original object output shape:", demo_result.shape)
print("Reloaded-from-MLflow output shape:", reloaded_result.shape)
print(reloaded_result.head())

In [2]:
assert np.allclose(demo_result['Weekly_Sales'], reloaded_result['Weekly_Sales'])
print("Value equality confirmed.")

Value equality confirmed.
